# تست augmentation نسخه‌ی ۳: استفاده از s2u_map واقعی (نه فرض ۸ اتم اول)

## چرا این نسخه؟
نسخه‌ی ۲ فرض کرده بود اتم‌های سلول واحد همان ۸ اندیس اول سوپرسل هستند
(`perm_unitcell = full_perm[:n_atoms]`). نتیجه‌ی اجرا نشان داد این فرض غلط است: برای
۲۲ از ۲۴ عملیات، اتم‌های مرجع به اندیس‌های خارج از `[0,8)` (مثلاً ۲۸ تا ۳۵) نگاشت شدند.

از تست‌های قبلی (notebook 18، v1-v3) می‌دانیم که `supercell.s2u_map` در Phonopy نشان
می‌دهد هر اتم سوپرسل با کدام نماینده (که خودش هم یک اندیس داخل سوپرسل است) متناظر است؛
نمایندگان واقعی برای این ساختار `3×3×1` در اندیس‌های `[0, 9, 18, 27, 36, 45, 54, 63]`
قرار دارند (طبق خروجی تست‌های notebook 18)، نه `[0,1,...,7]`.

## رفع باگ
به‌جای فرض کردن `full_perm[:n_atoms]`، این‌بار از `np.unique(s2u_map)` (نمایندگان واقعی
گروه‌ها) استفاده می‌کنیم تا بفهمیم سلول واحد واقعاً کدام ۸ اندیس از سوپرسل است، و permutation
بین خودِ همین ۸ اندیس را بر اساس `full_perm` استخراج می‌کنیم.

In [1]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("Installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.3/787.3 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 91.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 66.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 86.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [2]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS
import spglib

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
test_formula = common[0]
print(f"Test material: {test_formula}")

Test material: Cr2AlC


## بازسازی ماده با موتور واقعی Phonopy

In [3]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    n_images = n2_expected // len(unitcell.symbols)
    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue
    return best_dim, best_err


with open(band_yaml_files[test_formula]) as f:
    data = yaml.safe_load(f)

lattice = np.array(data['lattice'])
symbols = [p['symbol'] for p in data['points']]
frac_coords = np.array([p['coordinates'] for p in data['points']])
n_atoms = len(symbols)

unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

fc = parse_FORCE_CONSTANTS(str(fc_files[test_formula]))
n_supercell = fc.shape[1]

segment_len = data['segment_nqpoint'][0]
q_idx = min(segment_len // 2, len(data['phonon']) - 1)
q_test = data['phonon'][q_idx]['q-position']
real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n_supercell)
print(f"Discovered supercell_dim: {dim} (match error: {err:.6f})")

phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
phonon.force_constants = fc

phonon.run_qpoints([[0, 0, 0]])
original_freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
print(f"Original frequencies at Gamma (THz): {original_freqs}")

s2u_map = phonon.supercell.s2u_map
unitcell_sc_indices = np.unique(s2u_map)
print(f"\nReal unit-cell representative indices within the supercell: {unitcell_sc_indices}")
assert len(unitcell_sc_indices) == n_atoms

/tmp/ipykernel_58/3190667743.py:22: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
/kaggle/working/custom_lib/phonopy/api_phonopy.py:278: UserWarning: Warning: Point group symmetries of supercell and primitivecell are different.
  self._search_primitive_symmetry()


Discovered supercell_dim: (3, 3, 1) (match error: 0.000000)
Original frequencies at Gamma (THz): [-3.07950122e-03  2.43763264e-03  2.43763266e-03  2.99455968e+00
  2.99455968e+00  4.77698563e+00  4.77698563e+00  4.88722860e+00
  5.60689545e+00  5.60689545e+00  7.92996353e+00  7.99231856e+00
  7.99231856e+00  8.11675096e+00  8.11675096e+00  1.01656150e+01
  1.07428656e+01  1.14309913e+01  2.03112874e+01  2.03112874e+01
  2.03589956e+01  2.03589956e+01  2.04610455e+01  2.04984014e+01]

Real unit-cell representative indices within the supercell: [ 0  9 18 27 36 45 54 63]


/tmp/ipykernel_58/3190667743.py:59: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  original_freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])


## کشف عملیات تقارن با spglib

In [4]:
cell_tuple = (unitcell.cell, unitcell.scaled_positions, unitcell.numbers)
dataset = spglib.get_symmetry_dataset(cell_tuple, symprec=1e-3)

n_ops = len(dataset.rotations)
print(f"Space group: {dataset.international} (number {dataset.number})")
print(f"Number of symmetry operations: {n_ops}")

Space group: P6_3/mmc (number 194)
Number of symmetry operations: 24


## تابع‌های Transform (بدون تغییر نسبت به v2 — مشکل فقط در استخراج زیرمجموعه‌ی سلول واحد بود)

In [5]:
def find_supercell_permutation(supercell, R_cart, t_cart, symprec=1e-3):
    """
    Apply the cartesian symmetry operation (R_cart, t_cart) to every supercell atom
    position and find the matching original supercell atom (same element, same
    position modulo the supercell lattice). Returns a permutation array of length
    n_supercell, or None if no valid bijection exists.
    """
    sc_cart = supercell.positions
    sc_symbols = supercell.symbols
    sc_lattice = supercell.cell
    inv_lattice = np.linalg.inv(sc_lattice)

    n = len(sc_cart)
    transformed = sc_cart @ R_cart.T + t_cart

    perm = np.full(n, -1, dtype=int)
    for i in range(n):
        best_j, best_dist = -1, np.inf
        for j in range(n):
            if sc_symbols[j] != sc_symbols[i]:
                continue
            diff_cart = transformed[i] - sc_cart[j]
            diff_frac = diff_cart @ inv_lattice
            diff_frac = diff_frac - np.round(diff_frac)
            diff_cart_wrapped = diff_frac @ sc_lattice
            dist = np.linalg.norm(diff_cart_wrapped)
            if dist < best_dist:
                best_dist = dist
                best_j = j
        if best_dist > symprec:
            return None
        perm[i] = best_j

    if len(set(perm.tolist())) != n:
        return None
    return perm


def cartesian_rotation(R_frac, lattice):
    L = lattice
    return L.T @ R_frac @ np.linalg.inv(L.T)


def cartesian_translation(t_frac, lattice):
    return t_frac @ lattice


def transform_ifc_full(fc, perm_unitcell_into_sc_indices, perm_supercell, R_cart):
    """
    Transform the full IFC tensor (n_atoms, n_supercell, 3, 3) under a symmetry operation.
    perm_unitcell_into_sc_indices: for each ORIGINAL unit-cell atom i (0..n_atoms-1), the
        ROW index (0..n_atoms-1) in the IFC's first axis corresponding to the supercell
        atom that atom i's position maps to under this symmetry operation.
    perm_supercell: full (n_supercell,) permutation over supercell atom indices.
    """
    fc_permuted = fc[perm_unitcell_into_sc_indices][:, perm_supercell]
    fc_new = np.einsum('ab,nscd,db->nsac', R_cart, fc_permuted, R_cart)
    return fc_new

print("Transform functions ready")

Transform functions ready


## تابع جدید: نگاشت اندیس سوپرسل به اندیس محور اول IFC (۰ تا ۷)

IFC محور اولش طول `n_atoms=8` دارد و ترتیبش با ترتیب اتم‌های `unitcell` یکی است (نه ترتیب
دلخواه سوپرسل). برای هر اتم سلول واحد `i` (۰ تا ۷)، اندیس متناظرش در سوپرسل را از
`u2s_map` (یا معادل آن، اولین عضو `s2u_map==i` که خودش `unitcell_sc_indices[i]` است)
می‌گیریم، سپس وقتی تقارن این اندیس سوپرسل را به یک اندیس سوپرسل دیگر می‌برد، باید بفهمیم
آن اندیس سوپرسل دیگر متعلق به کدام عضو محور اول IFC (۰ تا ۷) است.

In [6]:
def supercell_index_to_unitcell_axis(sc_idx, s2u_map, unitcell_sc_indices):
    """
    Given a supercell atom index, return which unit-cell-axis row (0..n_atoms-1) it
    corresponds to, i.e. the position of its s2u_map representative within
    unitcell_sc_indices. Returns None if sc_idx's representative isn't a unit-cell
    representative itself (shouldn't happen for valid reps, but checked for safety).
    """
    rep = s2u_map[sc_idx]
    matches = np.where(unitcell_sc_indices == rep)[0]
    if len(matches) == 0:
        return None
    return matches[0]


# Build: for each unit-cell atom i (0..n_atoms-1), what supercell index does it occupy?
unitcell_axis_to_sc_idx = unitcell_sc_indices  # already in order 0..n_atoms-1 by np.unique

print(f"unitcell_axis_to_sc_idx: {unitcell_axis_to_sc_idx}")
print("supercell_index_to_unitcell_axis ready")

unitcell_axis_to_sc_idx: [ 0  9 18 27 36 45 54 63]
supercell_index_to_unitcell_axis ready


In [7]:
max_errors = []

for op_idx in range(n_ops):
    R = dataset.rotations[op_idx]
    t = dataset.translations[op_idx]
    R_cart = cartesian_rotation(R, lattice)
    t_cart = cartesian_translation(t, lattice)

    full_perm_i = find_supercell_permutation(phonon.supercell, R_cart, t_cart)
    if full_perm_i is None:
        print(f"  op {op_idx}: FAILED to find a valid supercell permutation")
        continue

    # for each original unit-cell atom (axis index 0..n_atoms-1), find where its
    # supercell position maps to under this operation, then convert that target
    # supercell index back into a unit-cell-axis row index
    perm_unitcell_axis = np.full(n_atoms, -1, dtype=int)
    valid = True
    for axis_i in range(n_atoms):
        sc_idx = unitcell_axis_to_sc_idx[axis_i]
        target_sc_idx = full_perm_i[sc_idx]
        target_axis = supercell_index_to_unitcell_axis(target_sc_idx, s2u_map, unitcell_sc_indices)
        if target_axis is None:
            valid = False
            break
        perm_unitcell_axis[axis_i] = target_axis

    if not valid:
        print(f"  op {op_idx}: could not resolve unit-cell axis mapping, skipping")
        continue

    fc_t = transform_ifc_full(fc, perm_unitcell_axis, full_perm_i, R_cart)
    phonon_t = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
    phonon_t.force_constants = fc_t
    phonon_t.run_qpoints([[0, 0, 0]])
    freqs_t = np.sort(phonon_t.get_qpoints_dict()['frequencies'][0])

    max_err = np.max(np.abs(original_freqs - freqs_t))
    max_errors.append(max_err)
    status = "OK" if max_err < 1e-4 else "MISMATCH"
    print(f"  op {op_idx}: max frequency error = {max_err:.8f} THz [{status}]")

print(f"\nSummary: {len(max_errors)} / {n_ops} operations tested successfully")
print(f"Worst error across all tested operations: {max(max_errors):.8f} THz" if max_errors else "No operations tested")

/tmp/ipykernel_58/273467493.py:36: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs_t = np.sort(phonon_t.get_qpoints_dict()['frequencies'][0])


  op 0: max frequency error = 0.00000000 THz [OK]
  op 1: max frequency error = 0.00000000 THz [OK]
  op 2: max frequency error = 0.00000000 THz [OK]
  op 3: max frequency error = 0.00000000 THz [OK]
  op 4: max frequency error = 0.00000000 THz [OK]
  op 5: max frequency error = 0.00000000 THz [OK]
  op 6: max frequency error = 0.00000000 THz [OK]
  op 7: max frequency error = 0.00000000 THz [OK]
  op 8: max frequency error = 0.00000000 THz [OK]
  op 9: max frequency error = 0.00000000 THz [OK]
  op 10: max frequency error = 0.00000000 THz [OK]
  op 11: max frequency error = 0.00000000 THz [OK]
  op 12: max frequency error = 0.00000000 THz [OK]
  op 13: max frequency error = 0.00000000 THz [OK]
  op 14: max frequency error = 0.00000000 THz [OK]
  op 15: max frequency error = 0.00000000 THz [OK]
  op 16: max frequency error = 0.00000000 THz [OK]
  op 17: max frequency error = 0.00000000 THz [OK]
  op 18: max frequency error = 0.00000000 THz [OK]
  op 19: max frequency error = 0.00000000

## نتیجه‌گیری این تست

اگر این‌بار همه‌ی ۲۴ عملیات با خطای فرکانس نزدیک صفر تأیید شدند، یعنی این روش (استخراج
permutation محور اول IFC از طریق `s2u_map` واقعی، نه فرض ساده‌ی ۸ اندیس اول) درست است و
آماده‌ایم تا روی کل دیتاست اعمالش کنیم.

**لطفاً کل خروجی این نوت‌بوک را برای من بفرستید.**